# Homework: a file transfer protocol, and how TCP performs

In the previous lab you measured throughput with a simple sender and receiver on a Mininet network.
In this homework, which you complete **on your own**, you will:

1. **Part 1:** design and implement a small FTP-like protocol to list and download files;
2. **Part 2:** measure how long downloads take for different file sizes, bandwidths and delays;
3. **Part 3:** explain your results.

**What to submit:** this notebook **with all the outputs** (run every cell) and your answers written in it, plus
`apps/protocol.py`, `apps/server.py` and `apps/client.py`.

> **Requirements:** as in the lab, this notebook must run on a Linux machine with Mininet installed, and Jupyter must
> run as **root**. Experiments with large delays take a while: start early.

## 0. Setup

The files the server shares are in `files/`: `small.bin`, `medium.bin` and `large.bin`.
The client saves what it downloads in `downloads/` (remember that Mininet hosts share your file system:
the client must not overwrite the originals).

The topology is the same as in the lab, `h1` (client, 10.0.0.1) — `s1` — `h2` (server, 10.0.0.2),
with the `h1 - s1` link shaped by `bw` (Mbit/s), `delay` (e.g. `"10ms"`, applied **in each direction**) and `loss`
(in %, applied **in each direction**). `build_net()` now also starts the server on `h2`. You do not need to modify
the cells of this section.

In [ ]:
import hashlib
import json
import os
import re
import statistics
import time

from mininet.net import Mininet
from mininet.topo import Topo
from mininet.node import OVSBridge
from mininet.link import TCLink
from mininet.log import setLogLevel
from mininet.clean import cleanup

assert os.geteuid() == 0, "Mininet needs root: start Jupyter with sudo"

setLogLevel("warning")
cleanup()

APPS = os.path.abspath("apps")
FILES = os.path.abspath("files")
DOWNLOADS = os.path.abspath("downloads")
FILE_NAMES = ["small.bin", "medium.bin", "large.bin"]
SERVER_IP = "10.0.0.2"

for name in FILE_NAMES:
    print(f"{name:12} {os.path.getsize(os.path.join(FILES, name)):>12,} bytes")

net = h1 = h2 = None

In [ ]:
class TwoHostTopo(Topo):
    def build(self, bw=None, delay=None, loss=None):
        h1 = self.addHost("h1", ip="10.0.0.1/24")
        h2 = self.addHost("h2", ip="10.0.0.2/24")
        s1 = self.addSwitch("s1")
        self.addLink(h1, s1, bw=bw, delay=delay, loss=loss)  # shaped link
        self.addLink(h2, s1)                                 # unconstrained link


def build_net(bw=None, delay=None, loss=None):
    # (Re)build the network and start the server on h2
    global net, h1, h2
    if net is not None:
        h2.cmd("pkill -f apps/server.py")
        net.stop()
    net = Mininet(topo=TwoHostTopo(bw=bw, delay=delay, loss=loss),
                  link=TCLink, switch=OVSBridge, controller=None)
    net.start()
    h1, h2 = net.get("h1", "h2")
    h2.cmd(f"python3 -u {APPS}/server.py > /tmp/server.log 2>&1 &")
    for _ in range(20):
        if ":20003" in h2.cmd("ss -ltn"):
            return
        time.sleep(0.25)
    raise RuntimeError("The server did not start:\n" + h2.cmd("cat /tmp/server.log"))


def client(*args, timeout=60):
    # Run the client on h1 with the given arguments and return its output
    return h1.cmd(f"timeout {timeout} python3 {APPS}/client.py {SERVER_IP} " + " ".join(args))


def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(1 << 20):
            h.update(chunk)
    return h.hexdigest()


def download(name, timeout=120, verbose=False):
    # Download one file. Returns the transfer time in seconds, or None if it
    # failed or did not finish within `timeout` seconds.
    path = os.path.join(DOWNLOADS, name)
    if os.path.exists(path):
        os.remove(path)
    out = client("get", name, timeout=timeout)
    if verbose:
        print(out.strip())
    m = re.search(r"^RESULT (\S+) (\d+) ([\d.]+)", out, re.M)
    if m is None:
        print(f"  {name}: failed or timed out after {timeout} s\n  {out.strip()}")
        return None
    if md5(path) != md5(os.path.join(FILES, name)):
        print(f"  {name}: the downloaded file is different from the original!")
        return None
    return float(m.group(3))


def ping_rtt(count=10):
    # Average RTT measured by ping from h1 to h2, in ms
    out = h1.cmd(f"ping -c {count} -i 0.2 {SERVER_IP}")
    m = re.search(r"= [\d.]+/([\d.]+)/", out)
    return float(m.group(1)) if m else None

## Part 1. An FTP-like protocol

Right now the server and the client in `apps/` only do a bulk transfer: as soon as the client connects, the server
pushes 1 MB of dummy data in `DATA` messages, followed by an `END` message. The messages are defined in
`apps/protocol.py`: each one has a 5-byte header (type and payload length) followed by the payload. Read the three
files before going on. Let's check that the skeleton works:

In [ ]:
build_net()
print(client())

Your job is to turn this into a (very) simplified FTP. The client must support two commands:

* `python3 client.py SERVER_IP list` prints the files available on the server (the content of `files/`);
* `python3 client.py SERVER_IP get FILE` downloads **one** file into `downloads/`.

Requirements:

1. Use the message format of `protocol.py`: add the message types you need, and do not change the header.
2. The server answers requests until the client closes the connection, and keeps running afterwards (it must be
   able to serve many clients, one after the other).
3. Asking for a file that does not exist gives an error message on the client, **not** a crash of the server or the client.
4. The server only serves the files in `files/`. A client must not be able to download anything else
   (e.g. `../apps/server.py` or `/etc/passwd`).
5. Files can be large: the server must send them chunk by chunk, and the client must write them to disk as they arrive.
6. Do not change the line printing `RESULT ...` in `client.py`: the notebook uses it to collect your measurements.

The `TODO` comments in the three files tell you where to start.

**NOTE: Our solution of this exercise is based on ~40 lines of additional code. Keep this in mind and keep your solution simple! And make sure to use python libraries for file system management:**

```python
open()
os.listdir()
os.path.getsize()
os.path.join()
os.path.isfile()
...
```

**Question 1.1** — Before writing code, briefly describe your protocol here: the list of message types (who sends them,
what their payload contains and how it is encoded) and the sequence of messages exchanged for a `list` and for a `get`
of an existing and of a missing file.

*Your answer here.*

### Testing your implementation

Every time you change the code, rebuild the network (this restarts the server) and run the tests below.

In [ ]:
build_net()

# list
out = client("list")
print(out)
assert all(name in out for name in FILE_NAMES), "list does not show all the files"

# get: each file, twice (the server must keep working after each client)
for name in FILE_NAMES * 2:
    t = download(name, verbose=True)
    assert t is not None, f"downloading {name} failed"

# a missing file
out = client("get", "missing.bin")
print(out)
assert "RESULT" not in out and "Traceback" not in out, "a missing file must give a clean error message"

# files outside of files/
with open("/tmp/secret.txt", "w") as f:
    f.write("you should not be able to download this")
for name in ["/tmp/secret.txt", os.path.relpath("/tmp/secret.txt", FILES)]:
    out = client("get", name)
    print(out)
    assert "RESULT" not in out, f"the server must refuse to send {name}"

# the server is still alive
assert download("small.bin") is not None
print(h2.cmd("cat /tmp/server.log"))
print("All tests passed!")

**Question 1.2** — Your protocol adds a header to every chunk of data. What fraction of the bytes on the wire is
protocol overhead for a download of `large.bin` with the server's `chunkSize`? Which other headers are added below
your protocol (TCP, IP, Ethernet), and roughly how much do they add?

*Your answer here.*

## Part 2. Measuring performance

We now measure the download time of each file while changing the network. `run_config()` rebuilds the network with
the given parameters, measures the RTT with `ping`, downloads each of the given files `reps` times and stores the
results in `results` (also saved to `results.json`, so you don't lose them if the kernel dies: reload them with
`results = json.load(open("results.json"))`). `show()` prints a summary table.

The **throughput** of a download is its size divided by its duration (connection setup included).

In [ ]:
results = []


def run_config(bw=None, delay=None, loss=None, files=FILE_NAMES, reps=3, timeout=120):
    build_net(bw=bw, delay=delay, loss=loss)
    rtt = ping_rtt()
    print(f"bw={bw} Mbit/s, delay={delay}, loss={loss}%: RTT {rtt} ms")
    for name in files:
        times = [download(name, timeout=timeout) for _ in range(reps)]
        results.append({"bw": bw, "delay": delay, "loss": loss, "rtt_ms": rtt, "file": name,
                        "size": os.path.getsize(os.path.join(FILES, name)), "times": times})
    with open("results.json", "w") as f:
        json.dump(results, f, indent=1)


def show(rows=None):
    rows = results if rows is None else rows
    print(f"{'bw':>5} {'delay':>6} {'loss':>5} {'RTT ms':>7} {'file':>11} {'time s (mean ± sd)':>20}"
          f" {'Mbit/s':>8} {'failed':>6}")
    for r in rows:
        ok = [t for t in r["times"] if t is not None]
        failed = len(r["times"]) - len(ok)
        if ok:
            mean = statistics.mean(ok)
            sd = statistics.stdev(ok) if len(ok) > 1 else 0
            tput = f"{r['size'] * 8 / mean / 1e6:8.2f}"
            time_s = f"{mean:9.3f} ± {sd:8.3f}"
        else:
            tput, time_s = f"{'-':>8}", f"{'-':>20}"
        print(f"{str(r['bw']):>5} {str(r['delay']):>6} {str(r['loss']):>5} {str(r['rtt_ms']):>7}"
              f" {r['file']:>11} {time_s:>20} {tput} {failed:>6}")

### 2.1 Baseline

First, with no constraint on the link (the throughput is only limited by your CPU):

In [ ]:
run_config()
show()

### 2.2 File size and bandwidth-delay product

Now vary the bandwidth and the delay. Recall that `delay` is added in **each** direction, so the RTT is about
twice the delay. For each configuration, compute the **bandwidth-delay product** (bandwidth × RTT, in bytes) and add it
to your analysis.

Run at least the configurations below (you are welcome to add more). This takes a few minutes.

In [ ]:
for bw in [10, 100]:
    for delay in ["1ms", "10ms", "50ms", "100ms"]:
        run_config(bw=bw, delay=delay)
show()

**Plots 2.2** — The cell below plots these results (`pip install matplotlib` if needed):

1. throughput as a function of the RTT, one line per file, one plot per bandwidth;
2. throughput as a function of the file size, one line per RTT, for `bw=100`.

The dashed horizontal line is the link bandwidth. Use these plots to answer the questions of Part 3.

In [ ]:
import matplotlib.pyplot as plt


def throughput(r):
    # Mean throughput of a result row in Mbit/s, or None if every download failed
    ok = [t for t in r["times"] if t is not None]
    return r["size"] * 8 / statistics.mean(ok) / 1e6 if ok else None


def latest(rows):
    # Keep only the last row for each (bw, delay, loss, file), in case a configuration was run twice
    return list({(r["bw"], r["delay"], r["loss"], r["file"]): r for r in rows}.values())


bdp_rows = latest([r for r in results if r["bw"] in (10, 100) and r["loss"] is None])

# 1. Throughput vs RTT, one line per file, one plot per bandwidth
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, bw in zip(axes, [10, 100]):
    for name in FILE_NAMES:
        pts = sorted((r["rtt_ms"], throughput(r)) for r in bdp_rows
                     if r["bw"] == bw and r["file"] == name and throughput(r) is not None)
        if pts:
            ax.plot(*zip(*pts), marker="o", label=name)
    ax.axhline(bw, color="gray", linestyle="--", label="link bandwidth")
    ax.set_title(f"bw = {bw} Mbit/s")
    ax.set_xlabel("RTT (ms)")
    ax.set_ylabel("throughput (Mbit/s)")
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
plt.show()

# 2. Throughput vs file size, one line per RTT, for bw=100
fig, ax = plt.subplots(figsize=(7, 4.5))
for delay in sorted({r["delay"] for r in bdp_rows if r["bw"] == 100}, key=lambda d: float(d.rstrip("ms"))):
    rows = [r for r in bdp_rows if r["bw"] == 100 and r["delay"] == delay and throughput(r) is not None]
    if not rows:
        continue
    rtt = statistics.mean(r["rtt_ms"] for r in rows)
    pts = sorted((r["size"], throughput(r)) for r in rows)
    ax.plot(*zip(*pts), marker="o", label=f"delay {delay} (RTT ≈ {rtt:.1f} ms)")
ax.axhline(100, color="gray", linestyle="--", label="link bandwidth")
ax.set_xscale("log")
ax.set_title("bw = 100 Mbit/s")
ax.set_xlabel("file size (bytes)")
ax.set_ylabel("throughput (Mbit/s)")
ax.set_ylim(bottom=0)
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Part 3. Questions

Answer using **your** measurements: quote numbers from your tables and refer to your plots.

**Question 3.1** — Look at `small.bin`. How does its download time change with the RTT, and how does it change with the
bandwidth? Express the download time as a number of RTTs, and explain where each of these round trips comes from
(think about the TCP handshake, your protocol's messages, and slow start).

*Your answer here.*

**Question 3.2** — Look at `large.bin`. How close does it get to the link bandwidth? Does the RTT matter as much as for
`small.bin`? Relate your answer to the bandwidth-delay product and to TCP's congestion window.

*Your answer here.*

**Question 3.3** — For a given configuration (bandwidth and RTT), how does the throughput depend on the file size? Estimate
from your measurements the file size beyond which a download is limited by the bandwidth rather than by the latency.
How does this size change with the bandwidth-delay product?

*Your answer here.*

**Question 3.4** — Which configuration has the largest bandwidth-delay product? Would a file of a few MB ever reach the
link bandwidth there? What would happen to the throughput of `large.bin` if the RTT were 500 ms?

*Your answer here.*

**Question 3.5 (open question)** — Suppose you had to design a TCP variant for a path with a high loss rate
(say 5%) that is **not** caused by congestion. What would you change in TCP to keep a high throughput? Consider,
for instance, how the sender reacts to a loss, how it detects and recovers from losses, and how it could tell
congestion from random loss. For each change, discuss what could go wrong: what would happen if the losses were in
fact caused by congestion, and how would your variant share a link with regular TCP flows? There is no single right
answer: we are looking for a well-argued proposal.

*Your answer here.*

In [ ]:
# Clean up when you are done
h2.cmd("pkill -f apps/server.py")
net.stop()

# Submission instructions

Please upload an extracted version in HTML of this notebook, as well as the complete code, to the TP's submission page. You should submit your work within one hour of the end of the lab. Late submissions will be graded accordingly to the course's policy.

Before submission, run the following command from inside a terminal:

```bash
jupyter nbconvert --to html notebook.ipynb
```

Then compress the folder:

```bash
tar -czvf submission.tar.gz file_transfer/
```

And upload it to the submission page.